# `07_requests.ipynb`

In [ ]:
# uv add requests
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'

res = requests.get(URL)


raw_data = res.text    # str -> Parsing 안된 데이터
data = res.json()  # dict -> Parsing 된 데이터 (해석됨, 활용 가능)

In [ ]:
# 1인당 1등 당첨 금액 = 'rnk1WnAmt'

data['data']['list'][0]['rnk1WnAmt']  # 1197258718

- 이번주 당첨 정보중 다음 데이터를 추출

```py
...
print(lucky)  # [2, 13, 18, 32, 38, 42]
print(bonus)  # 22
```

In [ ]:
# dict 도 for 로 순회가 가능하다!
d = {'a': 1, 'b': 2, 'c': 3}

for k, v in d.items():
    print(k, v)

In [ ]:
lucky = [1, 2, 3, 4, 5, 6]

my = [1, 2, 3, 4, 5, 6]


# 1: General
count = 0
for ball in lucky:
    if ball in my:
        count += 1

print(count)

# 2: Python 특화
len(set(lucky) & set(my))

In [ ]:
# Main Mission
# 랜덤하게 뽑은 번호 6개와, 실제 당첨번호를 비교하여
# 몇등인지 출력하는 프로그램. 완성하면
# (추가미션) 함수로 잘 만들기 -> 함수로 돌려서 1등 나올때까지 결과 기록

# 1등: 숫자 6개 같음
# 2등: 숫자 5개 같고 + 나머지 하나가 보너스번호
# 3등 ~ 5등: 숫자 5개, 4개, 3개 같음

# 랜덤번호 vs 실제 당첨번호
# -> 우선 고정번호 vs 실제 당첨번호

In [ ]:
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
res = requests.get(URL)
data = res.json() 
core_data = data['data']['list'][0]

# 실제 당첨 숫자
lucky = []

for k, v in core_data.items():
    # 아! 모든 로또번호랑 연결된 key에는 'tm' 글자가 들어있군!
    if 'tm' in k:
        # key에 'tm' 들어간 경우에만 해당 value(로또번호)를 추가한다!
        lucky.append(v)
# 보너스 번호
bonus = core_data['bnsWnNo'] 

In [ ]:
import random

# 내가 랜덤하게 뽑은 숫자
my = random.sample(range(1, 46), 6)


match_count = len(set(lucky) & set(my))

if match_count == 6:
    result = '1'
elif match_count == 5 and bonus in my:
    result = '2'
elif match_count == 5:
    result = '3'
elif match_count == 4:
    result = '4'
elif match_count == 3:
    result = '5'
else:
    result = '꽝'

print(result)


In [ ]:
import requests

# 현실 로또 당첨 번호를 API 에서 가져옴
def fetch_lotto_info():
    URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
    res = requests.get(URL)
    data = res.json() 
    core_data = data['data']['list'][0]

    lucky = []
    for k, v in core_data.items():
        if 'tm' in k:
            lucky.append(v)

    bonus = core_data['bnsWnNo']
    # 최종 return 값은 튜플 (1, 2)
    return lucky, bonus

In [ ]:
# 공주머니 2개랑 보너스를 넣으면 등수를 알려줌
def check_my_luck(my_nums, real_nums, bonus):
    match_count = len(set(my_nums) & set(real_nums))

    if match_count == 6:
        result = '1'
    elif match_count == 5 and bonus in my_nums:
        result = '2'
    elif match_count == 5:
        result = '3'
    elif match_count == 4:
        result = '4'
    elif match_count == 3:
        result = '5'
    else:
        result = '꽝'
    return result

In [ ]:
# 1. 정보 받기
lucky, bonus = fetch_lotto_info()

In [ ]:
import random

dashboard = {
    '1': 0, '2': 0, '3': 0,
    '4': 0, '5': 0, '꽝': 0,
}

# 대시보드에 1등 나온 횟수가 0번이면
while dashboard['1'] == 0:
    # 내번호 랜덤으로 뽑기
    my = random.sample(range(1, 46), 6)
    # 결과 비교하기
    result = check_my_luck(my, lucky, bonus)
    # 대시보드 기록
    dashboard[result] += 1

print(dashboard)


## API Key 관리
1. 터미널에 `uv add python-dotenv` 로 설치
2. 모든 키 파일은 `.env` 파일에 보관 (없으면 생성)
3. 소스코드에서는 `load_dotenv()` 와 `os.getenv()` 를 사용하여 불러옴

In [ ]:
import os
from dotenv import load_dotenv
import requests

# .env 파일 불러오기
load_dotenv()

# 불러온 파일에서 원하는 Key 꺼내기
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

In [ ]:
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'

# 인증 관련 헤더
headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

In [ ]:
URL = BASE_URL + NEWS_URL

# 쿼리 파라미터를 dict 로 작성
params = {
    'query': '엔화',
    'sort': 'sim',
    # 100개 기사를 모아서 (시작은 5개로)
    'display': 5,
}
res = requests.get(URL, headers=headers, params=params)

In [93]:
# 100개 기사를 모아서
# title에 <b> </b> 이상한 태그 없애기 (검색 필요)
# 조건: link URL이 naver 뉴스인 애들만 모아야 함. 100개가 안될 수 있음.
# 간략히 다음과 같은 모양으로 만들기
# news 변수 내용을 csv 로 export 하기
data = res.json()['items']

news = []

for item in data:
    # 2. link에 naver가 없으면 버림
    if 'naver' in item['link']:
        # 1. <b> 없앤걸로 title 교체
        item['title'] = item['title'].replace('<b>', '').replace('</b>', '').replace('&quot;', '')  # 문자열에서 1번 인자를 2번 인자로 교체
        new_item = {
            'title': item['title'],
            'link': item['link']
        }
        news.append(new_item)

# for one_news in news:
#     requests.get(one_news['link'])

news

[{'title': '엔화, 美 장기금리 상승에 1달러=159엔대 전반 하락 출발',
  'link': 'https://n.news.naver.com/mnews/article/003/0014151350?sid=104'},
 {'title': '“엔화 저평가됐다”…호주 2위 연기금이 달러 약세·엔화강세에 베팅한...',
  'link': 'https://n.news.naver.com/mnews/article/009/0005726477?sid=101'},
 {'title': '엔화 약세에 日銀 긴축 빨라지나…9월 금리인상론 급부상',
  'link': 'https://n.news.naver.com/mnews/article/018/0006358116?sid=101'},
 {'title': '엔화도 美국채도 가격 매력 커졌지만…일학개미 순매수는 글쎄',
  'link': 'https://n.news.naver.com/mnews/article/001/0016269394?sid=101'},
 {'title': '투기 아닌 실수요가 엔저 주도…日정부 개입 효과 ‘한계’',
  'link': 'https://n.news.naver.com/mnews/article/015/0005324740?sid=104'}]

In [ ]:
import csv

filednames = news[0].keys()

with open('./news.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=filednames)
    writer.writeheader()
    writer.writerows(news)

## Parsing
1. JSON 문자열 -> dict 로 해석
2. HTML 문자열 -> 구조화 필요 (`BeautifulSoup4`)

In [ ]:
# uv add beautifulsoup4
import requests
from bs4 import BeautifulSoup


def extract_naver_news(url):
    # 네이버 뉴스 아니면 에러 발생
    if 'n.news.naver.com' not in url:
        raise Exception('네이버 뉴스가 아닙니다') 

    res = requests.get(url)
    # res.text 를 해석 완료!
    soup = BeautifulSoup(res.text, 'html.parser')
    # 해석한 HTML에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
    news_text = soup.select_one('#dic_area').text.strip()
    return news_text


URL = 'https://n.news.naver.com/article/008/0005405342'
extract_naver_news(URL)


In [ ]:
# 1. 특정 주제로 Naver News 연관도 순으로 5개 뽑기
# 2. Naver 뉴스 링크를 통해서 본문만 추출하기
# 3. 최종 결과는
'''
news = [
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
]
'''